# Week 1 NumPy 向量化练习

- 作者：邓涵丹
- 日期：2026-07-06
- 来源：`暑期居家集训学习计划.md` → Week 1 → NumPy 核心
- 适用周次：Week 1
- 分类：NumPy
- 关键词：广播、z-score、余弦相似度、softmax、one-hot、向量化 k-NN
- 运行环境：Python 3.10+、NumPy

## 练习1：z-score 标准化

实现按列标准化，使得每列均值为0、标准差为1。关键点：使用 `keepdims=True` 保持维度以利用广播。

In [1]:

import numpy as np

data = np.random.randn(100, 5)

row_means = data.mean(axis=0, keepdims=True)   
row_std = data.std(axis=0, keepdims=True)    
data_normalized = (data - row_means) / row_std

# 验证
print("标准化后每列均值：", np.round(data_normalized.mean(axis=0), 6))  # 应接近 0
print("标准化后每列标准差：", np.round(data_normalized.std(axis=0), 6))  # 应接近 1

标准化后每列均值： [ 0. -0.  0. -0. -0.]
标准化后每列标准差： [1. 1. 1. 1. 1.]


## 练习2：余弦相似度矩阵

计算一组向量两两之间的余弦相似度。要求：先 L2 归一化，再通过矩阵乘法得到相似度矩阵。

In [2]:

import numpy as np

data = np.random.randn(100, 5)
row_norm=np.linalg.norm(data,axis=1, keepdims=True)
data_normalized = data / row_norm
row_dot = np.dot(data_normalized, data_normalized.T)

print("对角线元素（自身相似度）：", np.round(np.diag(row_dot), 6))

对角线元素（自身相似度）： [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1.]


## 练习3：数值稳定版 softmax

实现 softmax 函数，支持任意批次维度，并在计算指数前减去最大值以防止溢出。

In [3]:

import numpy as np
def softmax(x):
 d_max=np.max(d,axis=-1,keepdims=True)
 d_exp=np.exp(d-d_max)

 d_sum=np.sum(d_exp,axis=-1, keepdims=True)
 d_softmax=d_exp/d_sum
 return d_softmax
#示例
d = np.random.randn(3, 5)
data = softmax(d)
print("每行和是否为1：", np.round(data.sum(axis=-1), 6))

每行和是否为1： [1. 1. 1.]


## 练习4：one-hot 编码

将整数标签转换为 one-hot 矩阵。使用花式索引直接赋值，避免循环。

In [4]:

import numpy as np

def one_hot(labels, num_classes):

    N = labels.shape[0]
    result = np.zeros((N, num_classes), dtype=np.float32)
    result[np.arange(N), labels] = 1.0
    return result
# 验证 
labels = np.array([0, 2, 1, 2, 0])
num_classes = 3
oh = one_hot(labels, num_classes)
print("one-hot 结果：")
print(oh)

one-hot 结果：
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 0. 1.]
 [1. 0. 0.]]


## 练习5：向量化 k-NN 距离矩阵

计算查询集与数据库之间的欧氏距离矩阵，完全使用广播和矩阵运算，不使用 `for` 循环。

In [5]:

import numpy as np

def knn_distances(query, database):
    # 1. 计算L2平方和
    q_sq = np.sum(query ** 2, axis=1, keepdims=True)    # (Q, 1)
    d_sq = np.sum(database ** 2, axis=1, keepdims=True).T  # (1, N)
    # 2. 计算查询集与数据库的点积矩阵 (Q, N)
    dot_product = query @ database.T
    dist_square = q_sq + d_sq - 2 * dot_product
    # 3. 开平方；用maximum防止浮点误差导致的负数
    dist = np.sqrt(np.maximum(dist_square, 0))
    return dist

# 验证
Q, N, D = 5, 8, 10
query = np.random.randn(Q, D)
database = np.random.randn(N, D)
dist_matrix = knn_distances(query, database)
print("距离矩阵形状：", dist_matrix.shape)  # 应为 (5, 8)

距离矩阵形状： (5, 8)
